# 11. Checking predict.py

`predictions.jsonl` was written by `Narration/narration.ipynb` from the saved `magi_output.json`.
Here `predict.py` runs the whole system again, straight from the page images, and its answers are compared
with that file. It also scores `predict.py` on the 80 dev sequences.

Kaggle: GPU T4, internet on, task data as input.

In [ ]:
%pip install -q "transformers==4.49.0" sentencepiece

In [ ]:
import os
import json

DATA = None
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if "sequences.json" in files:
        DATA = root
print("data:", DATA)

REPO = "/kaggle/working/repo"
!rm -rf {REPO}
!git clone -q https://github.com/arpitparashar06/Advanced-task {REPO}
!git -C {REPO} log -1 --format="repo at commit %h (%ad)"

## test: predict.py from the images

In [ ]:
!python {REPO}/predict.py --data {DATA} --split test --out /kaggle/working/predict_test.jsonl

## development: predict.py from the images, then the scorer

In [ ]:
!python {REPO}/predict.py --data {DATA} --split development --out /kaggle/working/dev_predictions.jsonl
!python {DATA}/score.py --references {DATA}/development/labels.jsonl --predictions /kaggle/working/dev_predictions.jsonl --output /kaggle/working/dev_scores.json

## is it the same as predictions.jsonl?

In [ ]:
old = {}
for line in open(REPO + "/predictions.jsonl"):
    x = json.loads(line)
    old[x["sequence_id"]] = x["pages"]

new = {}
for line in open("/kaggle/working/predict_test.jsonl"):
    x = json.loads(line)
    new[x["sequence_id"]] = x["pages"]

same = 0
same_text = 0
for sid in old:
    old_text = [[x["text"] for x in page] for page in old[sid]]
    new_text = [[x["text"] for x in page] for page in new[sid]]
    if new[sid] == old[sid]:
        same += 1
    else:
        print("different:", sid, " lines before:", sum(len(p) for p in old[sid]), " lines now:", sum(len(p) for p in new[sid]))
    if new_text == old_text:
        same_text += 1

print()
print("test sequences exactly the same (text and speakers):", same, "of", len(old))
print("test sequences with the same text:", same_text, "of", len(old))